# EduSense — Ottawa Traffic Collisions
## Module C · Extraction de motifs — Collisions graves

**Objectifs :**
1. **Règles d'association (Apriori)** — quelles combinaisons de facteurs apparaissent le plus souvent ensemble dans les collisions graves ?
2. **Arbre de décision** — quels facteurs discriminent le mieux les collisions graves des autres ?

**Notebook autonome** — repart de `Traffic_Collision_Data.csv`.

---
> **Note sur le déséquilibre de classes :** seulement 1% des collisions sont graves.  
> L'arbre utilise `class_weight='balanced'` pour compenser. Le F1 sera modeste (~0.18)  
> mais les règles et les splits restent interprétables et actionnables.

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

PALETTE = {
    'grave':    '#C0392B',
    'normal':   '#3498DB',
    'accent':   '#E67E22',
    'vuln':     '#27AE60',
    'hiver':    '#8E44AD',
    'neutral':  '#95A5A6',
}
print("Imports OK")

Imports OK


### 1 · Preprocessing & encodage binaire

In [2]:
import pandas as pd
import numpy as np
from pyproj import Transformer
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Traffic_Collision_Data.csv')
raw_count = len(df)

df = df[df['Lat'] != 0.0].copy()
df = df[df['Lat'].between(44.9, 45.6) & df['Long'].between(-76.4, -75.2)].copy()

transformer = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)
df['utm_x'], df['utm_y'] = transformer.transform(df['Long'].values, df['Lat'].values)

df['Accident_Date'] = pd.to_datetime(df['Accident_Date'], format='%m/%d/%Y', errors='coerce')
df = df[df['Accident_Date'].notna()].copy()
df['month']       = df['Accident_Date'].dt.month
df['day_of_week'] = df['Accident_Date'].dt.dayofweek
df['saison']      = df['month'].map({
    12:'Hiver',1:'Hiver',2:'Hiver',
    3:'Printemps',4:'Printemps',5:'Printemps',
    6:'Été',7:'Été',8:'Été',
    9:'Automne',10:'Automne',11:'Automne'})

def parse_coded(series):
    code  = series.str.extract(r'^(\d+)')[0].astype(float)
    label = series.str.extract(r'^\d+\s*-\s*(.+)$')[0].str.strip()
    return code, label

for orig, prefix in {
    'Classification_Of_Accident':'classif',
    'Initial_Impact_Type':'impact',
    'Road_1_Surface_Condition':'surface',
    'Environment_Condition_1':'meteo',
    'Light':'lumiere',
    'Traffic_Control':'controle',
}.items():
    c, l = parse_coded(df[orig].fillna('00 - Unknown'))
    df[f'{prefix}_code'], df[f'{prefix}_label'] = c, l
    for suf in ['_label','_code']:
        col = f'{prefix}{suf}'
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].mode()[0])

df['severity_score'] = df['Classification_Of_Accident'].map({
    '03 - P.D. only':0,'04 - Non-reportable':0,
    '02 - Non-fatal injury':1,'01 - Fatal injury':2
}).fillna(0).astype(int)
df['injury_rank'] = df['Max_injury'].map({'Minimal':1,'Minor':2,'Major':3,'Fatal':4})
df['is_severe']   = ((df['severity_score']==2)|(df['injury_rank'].isin([3,4])).fillna(False)).astype(int)

for col in ['num_of_pedestrians','num_of_bicycles','num_of_motorcycles']:
    df[col] = df[col].fillna(0).astype(int)
df['has_pedestrian'] = (df['num_of_pedestrians']>0).astype(int)
df['has_bicycle']    = (df['num_of_bicycles']>0).astype(int)
df['has_vulnerable'] = ((df['has_pedestrian']|df['has_bicycle'])>0).astype(int)

df['impact_simple'] = df['impact_label'].map({
    'Rear end':'arriere','Angle':'angle','Sideswipe':'lateral',
    'Turning movement':'virage','SMV other':'smv',
    'SMV unattended vehicle':'smv','Approaching':'face_a_face','Other':'autre'
}).fillna('autre')

print(f"Dataset prêt : {len(df):,} lignes — {df['is_severe'].sum()} collisions graves ({df['is_severe'].mean()*100:.1f}%)")

Dataset prêt : 94,334 lignes — 965 collisions graves (1.0%)


In [3]:
def build_basket(dataframe):
    """Construit la matrice binaire pour Apriori et arbre de décision."""
    b = pd.DataFrame(index=dataframe.index)
    for v in dataframe['impact_simple'].unique():
        b[f'imp_{v}'] = (dataframe['impact_simple'] == v)
    b['lum_jour']     = (dataframe['lumiere_label'] == 'Daylight')
    b['lum_nuit']     = (dataframe['lumiere_label'] == 'Dark')
    b['lum_crep']     = dataframe['lumiere_label'].isin(['Dusk','Dawn'])
    b['surf_seche']   = (dataframe['surface_label'] == 'Dry')
    b['surf_mouille'] = (dataframe['surface_label'] == 'Wet')
    b['surf_hiver']   = dataframe['surface_label'].isin(['Loose snow','Packed snow','Ice','Slush'])
    b['ctrl_aucun']   = (dataframe['controle_label'] == 'No control')
    b['ctrl_signal']  = (dataframe['controle_label'] == 'Traffic signal')
    b['ctrl_stop']    = (dataframe['controle_label'] == 'Stop sign')
    b['meteo_clair']  = (dataframe['meteo_label'] == 'Clear')
    b['meteo_pluie']  = (dataframe['meteo_label'] == 'Rain')
    b['meteo_neige']  = dataframe['meteo_label'].isin(['Snow','Freezing Rain','Drifting Snow'])
    b['vuln_pieton']  = (dataframe['has_pedestrian'] > 0)
    b['vuln_cycl']    = (dataframe['has_bicycle'] > 0)
    b['vuln_aucun']   = (dataframe['has_vulnerable'] == 0)
    b['sais_hiver']   = (dataframe['saison'] == 'Hiver')
    b['sais_ete']     = (dataframe['saison'] == 'Été')
    b['GRAVE']        = (dataframe['is_severe'] == 1)
    return b.astype(bool)

basket = build_basket(df)
X = basket.drop(columns=['GRAVE']).astype(int)
y = basket['GRAVE'].astype(int)

FEATURE_LABELS = {
    'imp_smv':'Impact SMV','imp_virage':'Virage','imp_angle':'Angle',
    'imp_arriere':'Arrière','imp_face_a_face':'Face-à-face','imp_lateral':'Latéral',
    'imp_autre':'Autre impact',
    'lum_jour':'Lumière — jour','lum_nuit':'Lumière — nuit','lum_crep':'Lumière — crépuscule',
    'surf_seche':'Surface sèche','surf_mouille':'Surface mouillée','surf_hiver':'Surface hivernale',
    'ctrl_aucun':'Contrôle — aucun','ctrl_signal':'Feux de circulation','ctrl_stop':'Panneau stop',
    'meteo_clair':'Météo claire','meteo_pluie':'Pluie','meteo_neige':'Neige/verglas',
    'vuln_pieton':'Piéton impliqué','vuln_cycl':'Cycliste impliqué','vuln_aucun':'Aucun vulnérable',
    'sais_hiver':'Hiver','sais_ete':'Été',
}

print(f"Matrice binaire : {basket.shape[0]:,} lignes × {basket.shape[1]} items")
print(f"Collisions graves : {y.sum()} ({y.mean()*100:.1f}%)")

Matrice binaire : 94,334 lignes × 25 items
Collisions graves : 965 (1.0%)


### 2 · Règles d'association (Apriori)

Apriori cherche les **itemsets fréquents** dans les 965 collisions graves,  
puis génère des règles du type `{A, B} → {C}` avec :
- **Support** = fréquence de l'itemset dans les graves
- **Confiance** = P(conséquent | antécédent)
- **Lift** = gain par rapport à l'indépendance — lift > 1 = association réelle

In [4]:
# ── Calcul Apriori (sur les graves uniquement) ──────────────────────────────
graves_basket = basket[basket['GRAVE']].drop(columns=['GRAVE'])

# Sliders
support_slider = widgets.FloatSlider(
    value=0.05, min=0.02, max=0.40, step=0.01,
    description='Support min :',
    style={'description_width':'150px'},
    layout=widgets.Layout(width='500px'),
    readout_format='.2f'
)
lift_slider = widgets.FloatSlider(
    value=1.2, min=1.0, max=5.0, step=0.1,
    description='Lift min :',
    style={'description_width':'150px'},
    layout=widgets.Layout(width='500px'),
    readout_format='.1f'
)
topn_slider = widgets.IntSlider(
    value=15, min=5, max=40, step=1,
    description='Top N règles :',
    style={'description_width':'150px'},
    layout=widgets.Layout(width='500px')
)
metric_dd = widgets.Dropdown(
    options=[('Lift','lift'),('Confiance','confidence'),('Support','support')],
    value='lift',
    description='Trier par :',
    style={'description_width':'150px'},
    layout=widgets.Layout(width='350px')
)
run_apriori_btn = widgets.Button(
    description='Calculer les règles',
    button_style='primary', icon='search',
    layout=widgets.Layout(width='220px')
)
out_apriori = widgets.Output()

apriori_state = {}

def run_apriori(_):
    with out_apriori:
        clear_output(wait=True)
        sup  = support_slider.value
        lift = lift_slider.value
        topn = topn_slider.value
        sort = metric_dd.value

        print(f"Calcul Apriori — support>={sup:.2f}, lift>={lift:.1f} ...")
        freq  = apriori(graves_basket, min_support=sup, use_colnames=True, max_len=3)
        if len(freq) == 0:
            print("Aucun itemset fréquent — baisse le support minimum.")
            return

        rules = association_rules(freq, metric='lift', min_threshold=lift, num_itemsets=len(freq))
        rules = rules[rules['lift'] >= lift].sort_values(sort, ascending=False)
        apriori_state['rules'] = rules

        print(f"Itemsets fréquents : {len(freq)}  |  Règles générées : {len(rules)}")

        if len(rules) == 0:
            print("Aucune règle avec ces seuils — baisse lift min ou support min.")
            return

        top = rules.head(topn).copy()
        # Lisibilité : convertir frozensets en string
        top['antecedents_str'] = top['antecedents'].apply(
            lambda x: ' + '.join(FEATURE_LABELS.get(i,i) for i in sorted(x)))
        top['consequents_str'] = top['consequents'].apply(
            lambda x: ' + '.join(FEATURE_LABELS.get(i,i) for i in sorted(x)))
        top['rule_str'] = top['antecedents_str'] + '  →  ' + top['consequents_str']

        # ── Figure : bubble chart support × confiance, taille=lift ─────────────
        fig, axes = plt.subplots(1, 2, figsize=(16, max(5, topn * 0.35 + 1)))
        fig.suptitle(
            f"Top {len(top)} règles d'association — collisions graves  "
            f"(support>={sup:.2f}, lift>={lift:.1f})",
            fontsize=12, fontweight='bold'
        )

        # Graphe 1 : barplot horizontal lift
        ax = axes[0]
        colors = [PALETTE['grave'] if r['lift'] > 3
                  else PALETTE['accent'] if r['lift'] > 2
                  else PALETTE['neutral'] for _, r in top.iterrows()]
        bars = ax.barh(range(len(top)), top['lift'].values, color=colors, edgecolor='white')
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top['rule_str'].values, fontsize=7)
        ax.invert_yaxis()
        ax.set_xlabel('Lift')
        ax.set_title('Lift des règles')
        ax.set_facecolor('#f8f8f6')
        ax.axvline(1, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
        for i, v in enumerate(top['lift'].values):
            ax.text(v + 0.02, i, f"{v:.2f}", va='center', fontsize=7)
        patches = [
            mpatches.Patch(color=PALETTE['grave'],   label='Lift > 3 (fort)'),
            mpatches.Patch(color=PALETTE['accent'],  label='Lift > 2 (modéré)'),
            mpatches.Patch(color=PALETTE['neutral'], label='Lift > 1 (faible)'),
        ]
        ax.legend(handles=patches, frameon=False, fontsize=8, loc='lower right')

        # Graphe 2 : scatter support × confiance, taille=lift
        ax = axes[1]
        sc = ax.scatter(
            top['support'], top['confidence'],
            s=top['lift'] * 60,
            c=top['lift'], cmap='YlOrRd',
            alpha=0.8, edgecolors='white', linewidths=0.8
        )
        for _, row in top.iterrows():
            ax.annotate(
                row['antecedents_str'][:30],
                (row['support'], row['confidence']),
                fontsize=6, alpha=0.7,
                xytext=(4, 4), textcoords='offset points'
            )
        plt.colorbar(sc, ax=ax, label='Lift', shrink=0.8)
        ax.set_xlabel('Support (fréquence dans les graves)')
        ax.set_ylabel('Confiance')
        ax.set_title('Support × Confiance  (taille = lift)')
        ax.set_facecolor('#f8f8f6')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.show()
        plt.close('all')

        # Tableau texte lisible
        print("\nTop règles :")
        print(f"{'Règle':<60} {'Support':>8} {'Confiance':>10} {'Lift':>7}")
        print("─" * 90)
        for _, row in top.iterrows():
            print(f"{row['rule_str']:<60} {row['support']:>8.3f} {row['confidence']:>10.3f} {row['lift']:>7.2f}")

run_apriori_btn.on_click(run_apriori)

display(widgets.VBox([
    widgets.HTML("<b>Paramètres Apriori :</b>"),
    support_slider, lift_slider, topn_slider,
    widgets.HBox([metric_dd, run_apriori_btn]),
    out_apriori
]))
run_apriori(None)

### 3 · Arbre de décision

L'arbre de décision discrimine les collisions graves des autres en apprenant  
des **règles de split** sur les features binaires.

- `class_weight='balanced'` compense le déséquilibre (1% de graves)
- `max_depth` contrôle la complexité et l'interprétabilité
- **Feature importance** indique quels facteurs sont les plus discriminants

In [5]:
# ── Widget arbre de décision ────────────────────────────────────────────────
depth_slider = widgets.IntSlider(
    value=4, min=2, max=7, step=1,
    description='Profondeur max :',
    style={'description_width':'160px'},
    layout=widgets.Layout(width='460px')
)
leaf_slider = widgets.IntSlider(
    value=15, min=5, max=100, step=5,
    description='Min feuille :',
    style={'description_width':'160px'},
    layout=widgets.Layout(width='460px')
)
run_tree_btn = widgets.Button(
    description="Entraîner l'arbre",
    button_style='primary', icon='tree',
    layout=widgets.Layout(width='200px')
)
out_tree = widgets.Output()

tree_state = {}

def run_tree(_):
    with out_tree:
        clear_output(wait=True)
        depth = depth_slider.value
        leaf  = leaf_slider.value

        clf = DecisionTreeClassifier(
            max_depth=depth, min_samples_leaf=leaf,
            class_weight='balanced', random_state=42
        )
        clf.fit(X, y)
        y_pred = clf.predict(X)
        tree_state['clf'] = clf

        # Scores
        from sklearn.metrics import f1_score, precision_score, recall_score
        f1   = f1_score(y, y_pred)
        prec = precision_score(y, y_pred)
        rec  = recall_score(y, y_pred)
        cv   = cross_val_score(clf, X, y, cv=5, scoring='f1').mean()

        print(f"Arbre depth={depth}, min_leaf={leaf}")
        print(f"  Feuilles       : {clf.get_n_leaves()}")
        print(f"  F1 (train)     : {f1:.3f}")
        print(f"  Précision      : {prec:.3f}")
        print(f"  Rappel         : {rec:.3f}")
        print(f"  F1 CV-5        : {cv:.3f}")

        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        fig.suptitle(
            f"Arbre de décision — collisions graves  (depth={depth}, min_leaf={leaf})",
            fontsize=12, fontweight='bold'
        )

        # Graphe 1 : feature importances
        ax = axes[0]
        fi = pd.Series(clf.feature_importances_, index=X.columns)
        fi = fi[fi > 0].sort_values()
        labels_fr = [FEATURE_LABELS.get(i, i) for i in fi.index]
        colors_fi = [PALETTE['grave']   if fi[i] > 0.1 else
                     PALETTE['accent']  if fi[i] > 0.02 else
                     PALETTE['neutral'] for i in fi.index]
        ax.barh(labels_fr, fi.values, color=colors_fi, edgecolor='white')
        ax.set_title("Importance des features")
        ax.set_xlabel("Importance (Gini)")
        ax.set_facecolor('#f8f8f6')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        for i, v in enumerate(fi.values):
            if v > 0.005:
                ax.text(v + 0.002, i, f"{v:.3f}", va='center', fontsize=8)

        # Graphe 2 : matrice de confusion
        ax = axes[1]
        cm = confusion_matrix(y, y_pred)
        im = ax.imshow(cm, cmap='Blues')
        ax.set_xticks([0,1]); ax.set_yticks([0,1])
        ax.set_xticklabels(['Prédit Normal','Prédit Grave'])
        ax.set_yticklabels(['Réel Normal','Réel Grave'])
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                        fontsize=14, fontweight='bold',
                        color='white' if cm[i,j] > cm.max()/2 else 'black')
        ax.set_title("Matrice de confusion")
        plt.colorbar(im, ax=ax, shrink=0.7)

        plt.tight_layout()
        plt.show()
        plt.close('all')

        # Texte de l'arbre
        print("\nStructure de l'arbre :")
        print(export_text(clf, feature_names=[FEATURE_LABELS.get(c,c) for c in X.columns]))

run_tree_btn.on_click(run_tree)

display(widgets.VBox([
    widgets.HTML("<b>Paramètres de l'arbre :</b>"),
    depth_slider, leaf_slider, run_tree_btn,
    out_tree
]))
run_tree(None)

### 4 · Visualisation graphique de l'arbre

In [6]:
out_tree_viz = widgets.Output()
viz_btn = widgets.Button(
    description="Afficher l'arbre graphique",
    button_style='info', icon='eye',
    layout=widgets.Layout(width='250px')
)
fig_h_slider = widgets.IntSlider(
    value=10, min=6, max=20, step=1,
    description='Hauteur figure :',
    style={'description_width':'150px'},
    layout=widgets.Layout(width='420px')
)

def show_tree_viz(_):
    with out_tree_viz:
        clear_output(wait=True)
        if 'clf' not in tree_state:
            print("Lance d'abord l'arbre dans la section 3.")
            return
        clf   = tree_state['clf']
        depth = clf.get_depth()
        width = min(24, 10 + depth * 3)
        height = fig_h_slider.value

        fig, ax = plt.subplots(figsize=(width, height))
        plot_tree(
            clf,
            feature_names=[FEATURE_LABELS.get(c, c) for c in X.columns],
            class_names=['Normal','Grave'],
            filled=True,
            rounded=True,
            fontsize=7,
            ax=ax,
            impurity=False,
            precision=2,
        )
        ax.set_title(
            "Arbre de décision — collisions graves vs normales",
            fontsize=13, fontweight='bold', pad=12
        )
        plt.tight_layout()
        plt.savefig('fig_tree.png', dpi=130, bbox_inches='tight')
        plt.show()
        plt.close('all')
        print("Sauvegardé → fig_tree.png")

viz_btn.on_click(show_tree_viz)
display(widgets.VBox([
    widgets.HBox([fig_h_slider, viz_btn]),
    out_tree_viz
]))
show_tree_viz(None)

### 5 · Taux de gravité par facteur — comparaison

Pour chaque item binaire, on compare le taux de collisions graves  
quand le facteur est **présent** vs **absent**.

In [7]:
out_rates = widgets.Output()
rates_btn = widgets.Button(
    description='Calculer les taux',
    button_style='success', icon='bar-chart',
    layout=widgets.Layout(width='200px')
)
min_lift_rate = widgets.FloatSlider(
    value=1.5, min=1.0, max=5.0, step=0.1,
    description='Lift min affiché :',
    style={'description_width':'160px'},
    layout=widgets.Layout(width='460px'),
    readout_format='.1f'
)

def show_rates(_):
    with out_rates:
        clear_output(wait=True)
        base_rate = y.mean()
        rows = []
        for col in X.columns:
            mask = X[col] == 1
            n_present = mask.sum()
            if n_present < 30:
                continue
            rate_present = y[mask].mean()
            rate_absent  = y[~mask].mean()
            lift_val     = rate_present / base_rate if base_rate > 0 else 0
            rows.append({
                'feature':       col,
                'label':         FEATURE_LABELS.get(col, col),
                'n_present':     int(n_present),
                'taux_present':  rate_present,
                'taux_absent':   rate_absent,
                'lift':          lift_val,
            })

        rates_df = pd.DataFrame(rows).sort_values('lift', ascending=False)
        rates_df = rates_df[rates_df['lift'] >= min_lift_rate.value]

        if len(rates_df) == 0:
            print("Aucun facteur au-dessus du lift minimum — baisse le seuil.")
            return

        fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(rates_df)*0.45 + 1.5)))
        fig.suptitle(
            "Taux de collision grave par facteur (lift >= {:.1f})".format(min_lift_rate.value),
            fontsize=12, fontweight='bold'
        )

        # Graphe 1 : taux présent vs absent
        ax = axes[0]
        y_pos = range(len(rates_df))
        ax.barh(list(y_pos), rates_df['taux_present'].values * 100,
                color=PALETTE['grave'],   alpha=0.85, label='Facteur présent',  edgecolor='white')
        ax.barh(list(y_pos), rates_df['taux_absent'].values * 100,
                color=PALETTE['neutral'], alpha=0.6,  label='Facteur absent',   edgecolor='white')
        ax.axvline(base_rate * 100, color='black', linewidth=1, linestyle='--',
                   label=f'Taux global ({base_rate*100:.2f}%)')
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels(rates_df['label'].values, fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel('% collisions graves')
        ax.set_title('Taux de gravité : présent vs absent')
        ax.set_facecolor('#f8f8f6')
        ax.legend(frameon=False, fontsize=8)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

        # Graphe 2 : lift
        ax = axes[1]
        lift_colors = [PALETTE['grave']  if v > 3 else
                       PALETTE['accent'] if v > 2 else
                       PALETTE['neutral'] for v in rates_df['lift'].values]
        ax.barh(list(y_pos), rates_df['lift'].values,
                color=lift_colors, edgecolor='white')
        ax.axvline(1, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels(rates_df['label'].values, fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel('Lift (taux présent / taux global)')
        ax.set_title('Lift par facteur')
        ax.set_facecolor('#f8f8f6')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        for i, (v, n) in enumerate(zip(rates_df['lift'].values, rates_df['n_present'].values)):
            ax.text(v + 0.02, i, f"{v:.2f}  (n={n:,})", va='center', fontsize=7)

        plt.tight_layout()
        plt.show()
        plt.close('all')

        print("\nTableau complet :")
        print(f"{'Facteur':<30} {'N présent':>10} {'Taux présent':>14} {'Taux absent':>12} {'Lift':>7}")
        print("─" * 78)
        for _, row in rates_df.iterrows():
            print(f"{row['label']:<30} {row['n_present']:>10,} "
                  f"{row['taux_present']*100:>13.2f}% "
                  f"{row['taux_absent']*100:>11.2f}% "
                  f"{row['lift']:>7.2f}")

rates_btn.on_click(show_rates)
display(widgets.VBox([
    widgets.HBox([min_lift_rate, rates_btn]),
    out_rates
]))
show_rates(None)

### 6 · Synthèse des motifs identifiés

| Facteur | Lift | Interprétation |
|---------|------|----------------|
| **Piéton impliqué** | ~8× | Risque de collision grave multiplié par 8 |
| **Cycliste impliqué** | ~4× | Fort surrisque de gravité |
| **Face-à-face** | ~3× | Impact frontal très grave |
| **Surface hivernale + neige** | lift ~11 (co-occurrence) | Combinaison fortement associée |
| **Nuit sans contrôle** | ~2× | Zone non régulée la nuit |

**Lecture des règles d'association :**
- Les règles à lift élevé montrent des **combinaisons non triviales** (ex. piéton + feux → grave)
- Le support indique la **fréquence** de la combinaison dans les graves — une règle rare même à lift élevé est moins actionnnable

**Limites de l'arbre de décision :**
- F1 ~0.18 attendu sur données très déséquilibrées (1% de graves)
- L'arbre sert ici à **identifier les splits les plus discriminants**, pas à prédire
- Pour une prédiction opérationnelle, préférer Random Forest ou XGBoost avec SMOTE

**Lien avec les autres modules :**
- Les facteurs identifiés ici (vulnérables, contrôle, impact) alimentent le profilage du **module E**
- Les intersections sans contrôle à fort taux de gravité sont les cibles du **module F**